In [2]:
import matplotlib
matplotlib.use('tkAgg')
import matplotlib.pyplot as plt
import numpy
import scipy.constants as con

In [3]:
Grav = con.G #I thought the library would probably be more accurate than coding it myself
#Made this a dictionary because this seemed appropriate. No real functional reason for it just nice organization.
Planet_database = {
'Earth': {'mass': 5.9722e24, 'radius': 6.371e6},
'Mars' : {'mass': 6.4171e23, 'radius': 3.3895e6},
'Moon' : {'mass': 7.342e22, 'radius': 1.7374e6},
}
Mass_Planet = Planet_database['Earth']['mass'] #The mass of the planet in kilograms. Used Earth as the reference
Rad_Planet = Planet_database['Earth']['radius'] #You can swap these out to evaluate a different planet
#m = 1 The mass of the particle in kilograms. Negligible compared to earth
gamma = Grav * Mass_Planet

In [ ]:
#These define all the helper functions
def acceleration(x: float = 1.0 , y: float = 0.0, eps=1e-6):
    r_vect = [x, y]
    r = numpy.sqrt(x**2 + y**2) #I got sick of writing this out
    if r < eps:
        r = eps #This is to avoid enormous radii caused by floating point errors as e approaches 1
    a_x, a_y = [-n/r**3 for n in r_vect] #The nondimensionalized equation of motion from Boris' notes
    return a_x, a_y, r

def Energy(vx, vy, x, y):
    r = numpy.sqrt(x**2 + y**2)
    return (0.5 * (vx**2 + vy**2)) - (gamma/r)
    #kinetic - potential; taking m = 1 since w/o loss of generality since it makes no impact on dynamics

def L(x, y, vx, vy): #Angular momentum
    return x * vy - y * vx

def LRL_vect_Theoretical(vx, vy, L_z_Current, x, y):
    L = numpy.array([0, 0, L_z_Current])
    v = numpy.array([vx, vy, 0])
    r = numpy.sqrt(x**2 + y**2)
    return (numpy.cross(v, L) / gamma) - numpy.array([x/r, y/r, 0]) #This is simply the formula for the LRL Vector
    #The subtracted term is just the unit vector in the direction of the position vector
    #I used arrays because they have vector operations preprogrammed

def eccentricity(LRL_vector): 
    #Initially decided to use the energy instead of the LRL Vector since I figured energy would have less error creep in
    #numpy.sqrt(1 + (2*Energy*(L**2))/(gamma**2))
    return numpy.linalg.norm(LRL_vector)

def Period(E, tol=1e-6):
    if E >= -tol: #Theoretically this could be 0, but this couldn't work as e approaches 1
        return None  # Parabolic or unbound
    return numpy.sqrt(2) * numpy.pi * gamma / (abs(E)**1.5)

def theoretical_perigee(Lz, e): #eccentricity and angular momentum are the two parameters
    return (Lz**2 / gamma) / (1 + e)

def theoretical_apogee(Lz, e): 
    if e < 1: #For bounded orbits the apogee exists
        return (Lz**2 / gamma) / (1 - e)
    else: 
       print('This orbit is unbounded and no apogee exists.')
       return None
    
def rdotv(vx, vy, x, y): #This was essentially only for the report part of this
    v = numpy.array([vx, vy])
    r = numpy.array([x, y])
    return numpy.dot(r, v) #Just calculates the dot product of r and v

In [ ]:
def Kepler_Problem(x0: float = 0, y0: float = 7e6, vx0: float = 7550, vy0: float = 100, sleeptime: float = 0.01):
    #x and y are the initial position vector components
    #vx and vy are the initial velocity vector components
    #Sleeptime controls the pauses between plotting points. Tmax is the total number of timesteps in the simulation

    t = 0.0
    dt = 0.5
    eta = 0.02   #timestep optimization accuracy control factor
    dt_min = 0.01
    dt_max = 50.0

    #All the constants of motion
    E0 = Energy(vx0, vy0, x0, y0)
    Lz = L(x0, y0, vx0, vy0)
    LRL_vec = LRL_vect_Theoretical(vx0, vy0, Lz, x0, y0)
    e = eccentricity(LRL_vec)
    r_p = theoretical_perigee(Lz, e)
    r_a = theoretical_apogee(Lz, e)

    # Define Boris scaling based on r_p_phys
    R0 = r_p
    T0 = numpy.sqrt(R0**3 / gamma)
    V0 = R0 / T0   

    #Printing all the important stuff
    print(f"Planet mass = {Mass_Planet:.3e} kg, radius = {Rad_Planet:.3e} m")
    print(f"Initial pos = ({x0:.3e}, {y0:.3e}), Initial Vel = ({vx0:.3e}, {vy0:.3e})")
    print(f"Angular Momentum = {Lz}")
    print(f"Theoretical LRL Vector = {LRL_vec[0]:.4e}, {LRL_vec[1]:.4e}")
    print(f"Eccentricity e = {e:.4f}")
    print(f"Perigee radius = {r_p:.3e} m")

    if r_p < Rad_Planet:
        print("Perigee inside planet radius. Continue simulation?") #asks if the user wants to continue the simulation even though a collison is likely
        s = input('y/n?')
        if s != 'y':
            return None #Stops the function from running
    if e < 1:
        print(f"Apogee radius = {r_a:.3e} m") #Apogee and period are in an if statement to print
        print(f"Period = {Period(E0):.3e} s or {(Period(E0)/3600):.3e} hours") #This is because there are cases in which they don't exist
        #Prints the period in both seconds and hours for convienience since seconds is trivial on this scale
    else:
        print("Unbound orbit (no apogee, no period).")

    iplot = 0
    iprint = 0
    t_max = Period(E0)/T0 if Period(E0) else 10 #We want to plot a single, nondimensionalized period, but if there is no period then we just set t_max to 10

    #Convert initial conditions to nondimensionalized counterparts
    x_nd = x0 / R0
    y_nd = y0 / R0
    vx_nd = vx0 / V0
    vy_nd = vy0 / V0
    dt_nd = dt / T0 #initial nondimensionalized timestep
    t_nd = 0.0
    a_x, a_y, r_nd = acceleration(x_nd, y_nd) #Initializes the nondimensionalized acceleration  
    
    #Scaling everything so that things are visible on the graph. 5*r_p seems to work well.
    scale = (2*r_a if e < 1 else 4*r_p) / R0
    print(f"Auto-adjusted scale to: {-scale:.1f} to {scale:.1f}")
    #Setup the plot
    plt.ion()  # turning interactive mode on
    fig, ax = plt.subplots()
    plt.axis('equal')
    ax.set_xlim(-scale, scale)
    ax.set_ylim(-scale, scale)
    circle = plt.Circle((0, 0), Rad_Planet/R0, fill=True, color='b') #Radius of the planet is nondimensionalized here
    ax.add_patch(circle) #Adds the circle that represents the planet
    ax.scatter(0, 0, color='black', s=10) #Center of the planet
    plt.grid('both')
    plt.xlabel('x / r_p (Dimensionless Rescaling)') # Abscissa label
    plt.ylabel('y / r_p (Dimensionless Rescaling)') # Ordinate label
    plt.title('Kepler Problem') # Title

    # marking the theoretical perigee and (maybe) apogee on the graph
    #Rescaled since the graph is nondimensionalized
    ax.scatter(r_p/R0, 0, color='green', s=60, label='Perigee')
    if r_a is not None:
        ax.scatter(-r_a/R0, 0, color='black', s=60, label='Apogee')
    
    #Didn't name it in the same way as the other rescalings since it would have been much more effort to retool and kept getting NameErrors

    #This is the nondimensionalized Velocity-Verlet Integrator Loop
    while (t_nd) < t_max: #t_nd * T0 will just yield the original maximum time bc I've gotten lazy at rescaling this stuff
        vx_half_nd, vy_half_nd = vx_nd + 0.5*a_x*dt_nd, vy_nd + 0.5*a_y*dt_nd #This gets the velocity at time delta t + dt
        x_nd, y_nd = x_nd + vx_half_nd*dt_nd, y_nd + vy_half_nd*dt_nd #Compute new positions
        a_xnew, a_ynew, r_nd = acceleration(x_nd, y_nd) #compute the new acceleration
        vx_nd, vy_nd = vx_half_nd + 0.5*a_xnew*dt_nd, vy_half_nd + 0.5*a_ynew*dt_nd #compute the velocities using the half time step velocity
        a_x, a_y = a_xnew, a_ynew #The new accelerations are now a_x and a_y so the loop doesnt have to recompute them
        
        if (r_nd * R0) <= Rad_Planet: #This is the best place to put it because it minimizes unneccessary computation
            print(f"Collision with planet surface at {(t*T0)/3600} hours") #This prints the time of collision in dimensionalized units
            break
        
        E_Current = Energy(vx_nd*V0, vy_nd*V0, x_nd*R0, y_nd*R0)  #Velocity FIRST, then position. Definitely caught this before it ruined my data
        L_Current = L(x_nd*R0, y_nd*R0, vx_nd*V0, vy_nd*V0)
        LRL_vect = LRL_vect_Theoretical(vx_nd * V0, vy_nd * V0, L_Current, x_nd*R0, y_nd*R0)

        #formula for adjusting the timestep on the fly
        dt = dt_nd*T0
        dt = min(max(dt, dt_min), dt_max) #you have no un-nondimensionalize then re-nondimensionalize here
        dt_nd = dt/T0

        if iplot == 500:
            plt.pause(sleeptime)
            ax.scatter(x_nd, y_nd, s=30, c='r')
            iplot = 0

        if iprint == 10: #this prints the dimensionful versions of these values even though the plot is dimensionless
            print(f"t = {t*T0:.3f}\nVelocities = {float(vx_nd * V0):.6f}, {float(vy_nd * V0):.6f}\nEnergy Total = {E_Current:.6f}\nZ-Comp Angular Momentum = {(L_Current):.6f}") 
            print(f"LRL_vect = ({LRL_vect[0]:.3f}, {LRL_vect[1]:.3f})")
            iprint = 0
        
        iplot += 1
        iprint += 1
        t_nd += dt_nd

Kepler_Problem(x0=15e6, y0=0, vx0=-5156.0, vy0=5154.0)

This orbit is unbounded and no apogee exists.
Planet mass = 5.972e+24 kg, radius = 6.371e+06 m
Initial pos = (1.500e+07, 0.000e+00), Initial Vel = (-5.156e+03, 5.154e+03)
Angular Momentum = 77310000000.0
Theoretical LRL Vector = -3.6830e-04, 1.0000e+00
Eccentricity e = 1.0000
Perigee radius = 7.497e+06 m
Unbound orbit (no apogee, no period).
Auto-adjusted scale to: -4.0 to 4.0


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


t = 0.000
Velocities = -5165.762068, 5153.990758
Energy Total = 523.027038
Z-Comp Angular Momentum = 77309999999.999939
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5174.668791, 5153.966201
Energy Total = 523.027376
Z-Comp Angular Momentum = 77309999999.999893
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5183.606238, 5153.926093
Energy Total = 523.027714
Z-Comp Angular Momentum = 77309999999.999908
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5192.574539, 5153.870273
Energy Total = 523.028051
Z-Comp Angular Momentum = 77309999999.999924
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5201.573829, 5153.798578
Energy Total = 523.028389
Z-Comp Angular Momentum = 77309999999.999908
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5210.604239, 5153.710844
Energy Total = 523.028726
Z-Comp Angular Momentum = 77309999999.999908
LRL_vect = (-0.000, 1.000)
t = 0.000
Velocities = -5219.665903, 5153.606905
Energy Total = 523.029063
Z-Comp Angular Momentum = 77309999999.99987